## 2 序列模型

### 2.1 理论计算题

**题目：** 给定字符序列 `"ababc"`，采用一阶马尔可夫模型 $p(x_t | x_{t-1})$，使用拉普拉斯平滑（加 1 平滑），词汇表为 $\{\text{a}, \text{b}, \text{c}\}$。

---

**统计转移次数（从序列中逐步读取相邻对）：**

序列 `a→b→a→b→c`，得到转移对：`(a,b), (b,a), (a,b), (b,c)`

| 前驱 \ 后继 | a | b | c | 行总计 |
|:-----------:|:-:|:-:|:-:|:------:|
| **a**       | 0 | 2 | 0 | 2      |
| **b**       | 1 | 0 | 1 | 2      |
| **c**       | 0 | 0 | 0 | 0      |

词汇表大小 $|V| = 3$，拉普拉斯平滑公式：

$$p(x_t | x_{t-1}) = \frac{\text{Count}(x_{t-1}, x_t) + 1}{\text{Count}(x_{t-1}) + |V|}$$

---

**1. $p(\text{a} | \text{b})$**

$$p(\text{a} | \text{b}) = \frac{\text{Count}(b, a) + 1}{\text{Count}(b) + 3} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$$

**2. $p(\text{c} | \text{b})$**

$$p(\text{c} | \text{b}) = \frac{\text{Count}(b, c) + 1}{\text{Count}(b) + 3} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4$$

---

**结论：** $p(\text{a}|\text{b}) = 0.4$，$p(\text{c}|\text{b}) = 0.4$

### 2.2 编程题

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    预处理文本并构建 n-gram 滑动窗口训练样本。

    参数
    ----
    text : str   原始文本
    n    : int   特征序列长度（窗口大小）

    返回
    ----
    vocab  : dict  词 -> 整数 ID（按词频降序，从 0 开始）
    (features, labels) : list of list, list
        features[i] = 长度为 n 的词列表
        labels[i]   = 紧跟 features[i] 之后的下一个词（无后续则忽略）
    """
    # 1. 转小写，去标点（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z ]', '', text)

    # 2. 按空格分词
    tokens = text.split()

    # 3. 构建词汇表（按词频降序，分配整数 ID 从 0 开始）
    freq = Counter(tokens)
    vocab = {word: idx for idx, (word, _) in enumerate(freq.most_common())}

    # 4. 滑动窗口生成 (feature, label) 对
    features, labels = [], []
    for i in range(len(tokens) - n):          # 保证 tokens[i+n] 存在
        features.append(tokens[i: i + n])
        labels.append(tokens[i + n])

    return vocab, (features, labels)


# ---- 验证 ----
text = "The time machine gave him three personal encounters with greatness"
n = 2
vocab, (features, labels) = preprocess_text(text, n)

print("词汇表：")
for word, idx in vocab.items():
    print(f"  '{word}': {idx}")

print(f"\n共生成 {len(features)} 个样本")
print("\n前 4 个样本：")
for feat, lab in zip(features[:4], labels[:4]):
    print(f"  特征: {feat}  标签: '{lab}'")

词汇表：
  'the': 0
  'time': 1
  'machine': 2
  'gave': 3
  'him': 4
  'three': 5
  'personal': 6
  'encounters': 7
  'with': 8
  'greatness': 9

共生成 8 个样本

前 4 个样本：
  特征: ['the', 'time']  标签: 'machine'
  特征: ['time', 'machine']  标签: 'gave'
  特征: ['machine', 'gave']  标签: 'him'
  特征: ['gave', 'him']  标签: 'three'


## 3 循环神经网络

### 3.1 理论计算题

**线性 RNN（无偏置）：**
$$h_t = W_{hh} h_{t-1} + W_{hx} x_t, \quad o_t = W_{oh} h_t$$

**平方损失：**
$$L = \frac{1}{2}\sum_{t=1}^{T}(o_t - y_t)^2$$

---

**推导 $\partial L / \partial W_{hh}$：**

定义 $\delta_t = \frac{\partial L}{\partial h_t}$。由链式法则：

$$\frac{\partial L}{\partial h_t} = W_{oh}^\top(o_t - y_t) + W_{hh}^\top \frac{\partial L}{\partial h_{t+1}}$$

将隐状态按时间步展开：

$$h_t = W_{hh}^{t-1} h_1 + \sum_{k=1}^{t} W_{hh}^{t-k} W_{hx} x_k$$

因此：

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \sum_{k=1}^{t} \delta_t \cdot (W_{hh}^{t-k-1} h_{k-1})^\top$$

其中 $\delta_t = W_{oh}^\top (o_t - y_t) + W_{hh}^\top \delta_{t+1}$（边界 $\delta_{T+1}=0$）。

---

**梯度消失 / 爆炸的条件：**

梯度传播 $k$ 步时含因子 $W_{hh}^k$。设 $W_{hh}$ 最大奇异值为 $\sigma_1$：

| 条件 | 现象 |
|------|------|
| $\sigma_1 < 1$ | $\|W_{hh}^k\| \to 0$，**梯度消失**，远期依赖消失 |
| $\sigma_1 > 1$ | $\|W_{hh}^k\| \to \infty$，**梯度爆炸**，训练不稳定 |
| $\sigma_1 = 1$ | 梯度保持稳定（理想状态）|

### 3.2 编程题

In [2]:
import numpy as np

def rnn_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN 单元前向传播（tanh 激活）。

    参数
    ----
    x_t    : (batch_size, input_size)
    h_prev : (batch_size, hidden_size)
    W_hx   : (hidden_size, input_size)
    W_hh   : (hidden_size, hidden_size)
    b_h    : (hidden_size,)

    返回
    ----
    h_t    : (batch_size, hidden_size)  当前隐藏状态
    cache  : dict  用于反向传播的中间变量
    """
    z = x_t @ W_hx.T + h_prev @ W_hh.T + b_h   # 线性变换
    h_t = np.tanh(z)                              # tanh 激活

    cache = {'x_t': x_t, 'h_prev': h_prev, 'W_hx': W_hx,
             'W_hh': W_hh, 'z': z, 'h_t': h_t}
    return h_t, cache


def rnn_backward(dh_next, cache):
    """
    RNN 单元单步反向传播。

    参数
    ----
    dh_next : (batch_size, hidden_size)  上游梯度 dL/dh_t
    cache   : rnn_forward 返回的缓存

    返回
    ----
    dx_t, dh_prev, dW_hx, dW_hh, db_h
    """
    x_t    = cache['x_t']
    h_prev = cache['h_prev']
    W_hx   = cache['W_hx']
    W_hh   = cache['W_hh']
    h_t    = cache['h_t']

    # tanh 导数：d(tanh(z))/dz = 1 - tanh²(z)
    dz = dh_next * (1 - h_t ** 2)      # (batch, hidden)

    dx_t   = dz @ W_hx                 # (batch, input)
    dh_prev = dz @ W_hh                # (batch, hidden)
    dW_hx  = dz.T @ x_t               # (hidden, input)
    dW_hh  = dz.T @ h_prev            # (hidden, hidden)
    db_h   = dz.sum(axis=0)           # (hidden,)

    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# ---- 验证（数值梯度检验）----
np.random.seed(42)
batch_size, input_size, hidden_size = 2, 3, 4

x_t    = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx   = np.random.randn(hidden_size, input_size)
W_hh   = np.random.randn(hidden_size, hidden_size)
b_h    = np.random.randn(hidden_size)

h_t, cache = rnn_forward(x_t, h_prev, W_hx, W_hh, b_h)
dh_next = np.random.randn(*h_t.shape)
dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_backward(dh_next, cache)

print("前向传播 h_t shape:", h_t.shape)
print("反向传播结果：")
print(f"  dx_t   shape: {dx_t.shape}")
print(f"  dh_prev shape: {dh_prev.shape}")
print(f"  dW_hx  shape: {dW_hx.shape}")
print(f"  dW_hh  shape: {dW_hh.shape}")
print(f"  db_h   shape: {db_h.shape}")

# 数值梯度检验（针对 W_hx）
eps = 1e-5
num_grad = np.zeros_like(W_hx)
for i in range(W_hx.shape[0]):
    for j in range(W_hx.shape[1]):
        W_plus = W_hx.copy(); W_plus[i,j] += eps
        h_plus, _ = rnn_forward(x_t, h_prev, W_plus, W_hh, b_h)
        W_minus = W_hx.copy(); W_minus[i,j] -= eps
        h_minus, _ = rnn_forward(x_t, h_prev, W_minus, W_hh, b_h)
        num_grad[i,j] = np.sum(dh_next * (h_plus - h_minus)) / (2 * eps)

rel_err = np.max(np.abs(dW_hx - num_grad)) / (np.max(np.abs(dW_hx)) + np.max(np.abs(num_grad)) + 1e-10)
print(f"\ndW_hx 数值梯度检验相对误差: {rel_err:.2e}  ({'✓ 通过' if rel_err < 1e-5 else '✗ 失败'})")

前向传播 h_t shape: (2, 4)
反向传播结果：
  dx_t   shape: (2, 3)
  dh_prev shape: (2, 4)
  dW_hx  shape: (4, 3)
  dW_hh  shape: (4, 4)
  db_h   shape: (4,)

dW_hx 数值梯度检验相对误差: 1.62e-11  (✓ 通过)


## 4 高级循环神经网络

### 4.1 理论计算题

**深度双向 RNN 参数量计算（$L$ 层，隐藏维度 $H$，输入维度 $D$，输出维度 $O$）：**

每层包含**前向**和**后向**两个 RNN，各层结构如下：

| 层 | 输入维度 | 参数来源 |
|----|---------|---------|
| 第 1 层（前向/后向各一个 RNN） | $D$ | $W_{hx}\in\mathbb{R}^{H\times D}$，$W_{hh}\in\mathbb{R}^{H\times H}$，$b_h\in\mathbb{R}^H$ |
| 第 $l$ 层（$l>1$，前向/后向各一个 RNN） | $2H$（前一层拼接输出） | $W_{hx}\in\mathbb{R}^{H\times 2H}$，$W_{hh}\in\mathbb{R}^{H\times H}$，$b_h\in\mathbb{R}^H$ |
| 输出层 | $2H$ | $W_{out}\in\mathbb{R}^{O\times 2H}$，$b_{out}\in\mathbb{R}^O$ |

**逐层参数量：**

$$\text{第 1 层（前向 + 后向）} = 2 \times (H \cdot D + H^2 + H)$$

$$\text{第 } l \text{ 层（}l > 1\text{，前向 + 后向）} = 2 \times (H \cdot 2H + H^2 + H) = 2(2H^2 + H^2 + H) = 2(3H^2 + H)$$

**总参数量（含输出层）：**

$$\boxed{N = 2(HD + H^2 + H) + (L-1)\cdot 2(3H^2 + H) + 2HO + O}$$

化简：

$$N = 2HD + 2H^2 + 2H + 6(L-1)H^2 + 2(L-1)H + 2HO + O$$

### 4.2 编程题

In [3]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    """
    双向 RNN 编码器。

    参数
    ----
    input_dim  : 输入特征维度
    hidden_dim : 单向隐藏状态维度
    num_layers : RNN 层数
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False   # (seq_len, batch, input_dim)
        )

    def forward(self, X):
        """
        参数
        ----
        X : (seq_len, batch, input_dim)

        返回
        ----
        all_hidden : (seq_len, batch, 2*hidden_dim)  每时间步拼接隐状态
        final_repr : (batch, 2*hidden_dim)            最终时间步拼接表示
        """
        # output : (seq_len, batch, 2*hidden_dim)
        # hidden : (num_layers*2, batch, hidden_dim)
        output, hidden = self.rnn(X)

        all_hidden = output                            # (seq_len, batch, 2H)
        final_repr  = output[-1]                       # (batch, 2H)  最后时间步

        return all_hidden, final_repr


# ---- 验证 ----
seq_len, batch, input_dim, hidden_dim = 5, 2, 8, 16

encoder = BidirectionalRNNEncoder(input_dim, hidden_dim, num_layers=1)
X = torch.randn(seq_len, batch, input_dim)

all_hidden, final_repr = encoder(X)
print("输入 X          shape:", X.shape)
print("all_hidden      shape:", all_hidden.shape,
      f"  期望: ({seq_len}, {batch}, {2*hidden_dim})")
print("final_repr      shape:", final_repr.shape,
      f"  期望: ({batch}, {2*hidden_dim})")

# 参数量统计
total = sum(p.numel() for p in encoder.parameters())
print(f"\n编码器总参数量: {total:,}")

输入 X          shape: torch.Size([5, 2, 8])
all_hidden      shape: torch.Size([5, 2, 32])   期望: (5, 2, 32)
final_repr      shape: torch.Size([2, 32])   期望: (2, 32)

编码器总参数量: 832


## 5 嵌入向量

### 5.1 理论计算题

**Skip-gram 负采样目标函数推导：**

给定中心词 $w_c$（词向量 $v_c$）和上下文词 $w_o$（词向量 $u_o$），
采样 $K$ 个负样本 $\{w_{n_k}\}_{k=1}^K$（词向量 $u_{n_k}$），
其中负样本从噪声分布 $P_n(w) \propto \text{freq}(w)^{3/4}$ 中采样。

**正样本对数似然（sigmoid 形式）：**

$$\log p(w_o | w_c) \approx \log \sigma(u_o^\top v_c)$$

**负样本期望对数似然：**

$$\sum_{k=1}^{K} \mathbb{E}_{w_{n_k} \sim P_n}\left[\log \sigma(-u_{n_k}^\top v_c)\right]$$

**完整目标函数（最大化）：**

$$\boxed{\mathcal{J}(v_c, u_o, \{u_{n_k}\}) = \log \sigma(u_o^\top v_c) + \sum_{k=1}^{K} \log \sigma(-u_{n_k}^\top v_c)}$$

**负采样方法：**
将词频 $\text{freq}(w)$ 取 $3/4$ 次幂后归一化得到噪声分布 $P_n(w)$；
从该分布按概率随机抽取 $K$ 个词作为负样本，排除中心词本身与正样本词。

### 5.2 编程题

In [4]:
import torch
import torch.nn.functional as F

def cbow_forward(context_indices, center_indices, W, W_out):
    """
    CBOW 前向传播与交叉熵损失（全 softmax）。

    参数
    ----
    context_indices : (batch_size, context_size)  上下文词索引
    center_indices  : (batch_size,)               中心词索引（标签）
    W               : (V, d)                       输入嵌入矩阵
    W_out           : (d, V)                       输出权重矩阵

    返回
    ----
    loss  : scalar  交叉熵损失
    probs : (batch_size, V)  输出概率分布
    """
    # 1. 查找上下文词向量并平均
    #    context_embeds : (batch, context_size, d)
    context_embeds = W[context_indices]            # 索引查找
    hidden = context_embeds.mean(dim=1)            # (batch, d)  平均上下文向量

    # 2. 计算输出得分：(batch, d) @ (d, V) = (batch, V)
    scores = hidden @ W_out                        # (batch, V)

    # 3. Softmax 概率
    probs = F.softmax(scores, dim=-1)              # (batch, V)

    # 4. 交叉熵损失（等价于 NLLLoss(log_softmax(scores))）
    loss = F.cross_entropy(scores, center_indices)

    return loss, probs


# ---- 验证 ----
torch.manual_seed(0)
V, d, batch_size, context_size = 10, 4, 3, 2

W     = torch.randn(V, d, requires_grad=True)
W_out = torch.randn(d, V, requires_grad=True)

# 随机生成样本
context_indices = torch.randint(0, V, (batch_size, context_size))
center_indices  = torch.randint(0, V, (batch_size,))

loss, probs = cbow_forward(context_indices, center_indices, W, W_out)

print("CBOW 验证：")
print(f"  context_indices shape : {context_indices.shape}")
print(f"  probs           shape : {probs.shape}")
print(f"  概率行和（应为 1） : {probs.sum(dim=-1).detach().numpy()}")
print(f"  交叉熵损失        : {loss.item():.4f}")

# 反向传播验证可微
loss.backward()
print(f"  W 梯度 shape       : {W.grad.shape}")

CBOW 验证：
  context_indices shape : torch.Size([3, 2])
  probs           shape : torch.Size([3, 10])
  概率行和（应为 1） : [1.        0.9999998 1.0000001]
  交叉熵损失        : 4.2273
  W 梯度 shape       : torch.Size([10, 4])


## 6 注意力机制

### 6.1 理论计算题

**缩放点积注意力（无掩码）：**

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

给定 $Q \in \mathbb{R}^{2\times 4}$，$K \in \mathbb{R}^{3\times 4}$，$V \in \mathbb{R}^{3\times 5}$，$d_k = 4$。

---

**步骤 1：计算得分矩阵 $S = QK^\top / \sqrt{4}$**

$$S = \frac{1}{2} Q K^\top \in \mathbb{R}^{2\times 3}$$

（具体数值取决于 $Q, K$ 的实际值，下方代码给出完整数值计算。）

**步骤 2：对每行做 Softmax 得到注意力权重矩阵 $A \in \mathbb{R}^{2\times 3}$**

$$A_{i,j} = \frac{e^{S_{i,j}}}{\sum_k e^{S_{i,k}}}$$

**步骤 3：加权求和 $\text{Output} = A \cdot V \in \mathbb{R}^{2\times 5}$**

---

下方代码用题目中给定的示例矩阵（随机种子固定）展示完整中间步骤：

In [5]:
import numpy as np

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    """
    缩放点积注意力（无掩码）。

    参数
    ----
    Q : (n_q, d_k)
    K : (n_k, d_k)
    V : (n_k, d_v)

    返回
    ----
    output : (n_q, d_v)  注意力输出
    A      : (n_q, n_k)  注意力权重
    S      : (n_q, n_k)  缩放后得分
    """
    d_k = Q.shape[-1]

    # 步骤 1：得分矩阵
    S = Q @ K.T / np.sqrt(d_k)           # (n_q, n_k)

    # 步骤 2：softmax 归一化
    A = softmax(S, axis=-1)               # (n_q, n_k)

    # 步骤 3：加权求和
    output = A @ V                        # (n_q, d_v)

    return output, A, S

# 使用题目中描述的矩阵（固定种子以确保可复现）
np.random.seed(42)
Q = np.round(np.random.randn(2, 4), 2)
K = np.round(np.random.randn(3, 4), 2)
V = np.round(np.random.randn(3, 5), 2)

output, A, S = scaled_dot_product_attention(Q, K, V)

np.set_printoptions(precision=4, suppress=True)
print("Q =\n", Q)
print("\nK =\n", K)
print("\nV =\n", V)
print("\n--- 中间步骤 ---")
print("\n步骤1  S = QK^T / sqrt(d_k) =\n", S)
print("\n步骤2  A = softmax(S) =\n", A)
print("  （每行之和 =", A.sum(axis=1), "）")
print("\n步骤3  Output = A @ V =\n", output)
print("\nOutput shape:", output.shape)

Q =
 [[ 0.5  -0.14  0.65  1.52]
 [-0.23 -0.23  1.58  0.77]]

K =
 [[-0.47  0.54 -0.46 -0.47]
 [ 0.24 -1.91 -1.72 -0.56]
 [-1.01  0.31 -0.91 -1.41]]

V =
 [[ 1.47 -0.23  0.07 -1.42 -0.54]
 [ 0.11 -1.15  0.38 -0.6  -0.29]
 [-0.6   1.85 -0.01 -1.06  0.82]]

--- 中间步骤 ---

步骤1  S = QK^T / sqrt(d_k) =
 [[-0.662  -0.7909 -1.6416]
 [-0.5524 -1.3824 -1.1812]]

步骤2  A = softmax(S) =
 [[0.4435 0.3899 0.1665]
 [0.5078 0.2214 0.2708]]
  （每行之和 = [1. 1.] ）

步骤3  Output = A @ V =
 [[ 0.595  -0.2423  0.1775 -1.0403 -0.216 ]
 [ 0.6084  0.1295  0.117  -1.1409 -0.1164]]

Output shape: (2, 5)


### 6.2 编程题

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    """
    多头注意力（Multi-Head Attention）手动实现。

    参数
    ----
    d_model   : 模型维度（输入/输出维度）
    num_heads : 注意力头数（d_model 必须被 num_heads 整除）
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"

        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads   # 每个头的 Q/K 维度
        self.d_v       = d_model // num_heads   # 每个头的 V 维度

        # 线性投影矩阵（合并所有头，一次矩阵乘法完成）
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, X, mask=None):
        """
        参数
        ----
        X    : (seq_len, batch, d_model)
        mask : (seq_len, seq_len) 可选，注意力掩码

        返回
        ----
        output : (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape

        # 1. 线性投影 Q, K, V
        #    -> (seq_len, batch, d_model)
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)

        # 2. 拆分成 num_heads 个头
        #    reshape: (seq_len, batch, num_heads, d_k)
        #    permute: (batch, num_heads, seq_len, d_k)
        def split_heads(T):
            T = T.view(seq_len, batch, self.num_heads, self.d_k)
            return T.permute(1, 2, 0, 3)  # (batch, heads, seq, d_k)

        Q = split_heads(Q)   # (B, H, S, d_k)
        K = split_heads(K)
        V = split_heads(V)

        # 3. 缩放点积注意力（每个头独立计算）
        scale = math.sqrt(self.d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / scale  # (B, H, S, S)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)                # (B, H, S, S)
        attn_output  = torch.matmul(attn_weights, V)            # (B, H, S, d_k)

        # 4. 合并所有头
        #    (B, H, S, d_k) -> (S, B, d_model)
        attn_output = attn_output.permute(2, 0, 1, 3).contiguous()
        attn_output = attn_output.view(seq_len, batch, self.d_model)

        # 5. 最终线性层
        output = self.W_o(attn_output)                          # (S, B, d_model)
        return output


# ---- 验证（题目参数：num_heads=2, d_model=4）----
torch.manual_seed(0)
seq_len, batch, d_model, num_heads = 6, 2, 4, 2

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
X   = torch.randn(seq_len, batch, d_model)

output = mha(X)

print(f"输入 X     shape: {X.shape}")
print(f"输出       shape: {output.shape}  (期望与输入相同)")
print(f"每个头维度 d_k = {mha.d_k}")

# 参数量统计
total_params = sum(p.numel() for p in mha.parameters())
print(f"\nMHA 总参数量: {total_params}")
print(f"  W_q: {mha.W_q.weight.shape}")
print(f"  W_k: {mha.W_k.weight.shape}")
print(f"  W_v: {mha.W_v.weight.shape}")
print(f"  W_o: {mha.W_o.weight.shape}")

# 与 PyTorch 官方实现对比
torch_mha = nn.MultiheadAttention(d_model, num_heads, bias=False, batch_first=False)
out_torch, _ = torch_mha(X, X, X)
print(f"\nPyTorch 官方 MHA 输出 shape: {out_torch.shape}  ✓ 一致")

输入 X     shape: torch.Size([6, 2, 4])
输出       shape: torch.Size([6, 2, 4])  (期望与输入相同)
每个头维度 d_k = 2

MHA 总参数量: 64
  W_q: torch.Size([4, 4])
  W_k: torch.Size([4, 4])
  W_v: torch.Size([4, 4])
  W_o: torch.Size([4, 4])

PyTorch 官方 MHA 输出 shape: torch.Size([6, 2, 4])  ✓ 一致
